# Unified Lab 3: The Architecture of Evolution

## The Scenario
Welcome to **Synthetica**, an AI research lab specializing in evolving dynamic computational structures. In Module 2, you used Genetic Algorithms (GAs) to optimize fixed-length lists of parameters. But what if you don't know the size or shape of the solution you are looking for in advance? What if the architecture itself needs to dynamically grow and evolve?

In this lab, you will explore **Genetic Programming (GP)**. Instead of evolving simple arrays of numbers, you will evolve executable code in the form of hierarchical **Syntax Trees**.

Your objective is to build a **Symbolic Regression** engine from scratch: an AI capable of discovering hidden mathematical formulas directly from raw data by organically evolving its own equations!

* **Milestone 1:** Build the Syntax Tree evaluation engine and MSE fitness function.
* **Milestone 2:** Implement the genetic operators (Subtree Crossover and Node Mutation).
* **Milestone 3:** Assemble the GP Lifecycle to crack a hidden dataset.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/unified-labs/blob/main/the_genetic_programmer/Module3-Lab.ipynb)

## Milestone 1: Set Up

Run this cell first.

In [ ]:
# MILESTONE 1 SETUP - RUN THIS CELL FIRST

class Node:
    def __init__(self, value, left=None, right=None):
        self.value = value  # Can be an operator ('+', '-', '*', '/'), a variable ('x'), or a number
        self.left = left
        self.right = right

# A hidden dataset generated by the formula: y = (x * 2) + 1
dataset = [(1, 3), (2, 5), (3, 7), (4, 9), (5, 11)]

# Let's manually build a Syntax Tree representing: (x * 2) + 1
#       +
#      / \
#     * 1
#    / \
#   x   2
node_x = Node('x')
node_2 = Node(2)
node_mult = Node('*', left=node_x, right=node_2)
node_1 = Node(1)
test_root = Node('+', left=node_mult, right=node_1)

print("Synthetica GP Engine Initialized.")
print("Test Tree and Dataset loaded.")

## Milestone 1: The Syntax Tree (Symbolic Regression)
**Objective:** In Genetic Programming (GP), a "chromosome" is not a list of numbers; it is an executable program represented as a **Syntax Tree**. Internal nodes act as mathematical operators (e.g., `+`, `*`), and leaf nodes act as terminals (variables like `x` or constants like `2`).

Your first task is to build the execution engine for these trees. You will write a recursive function to evaluate a tree for a given input `x`. Then, you will write a function to calculate the tree's **Mean Squared Error (MSE)** against a dataset. In Symbolic Regression, MSE serves as the fitness function—a lower MSE means the tree closer approximates the true hidden formula!

### The Architect's Blueprint
*Write the `evaluate_tree(node, x)` and `calculate_mse(root_node, dataset)` functions.*

* **1. Evaluate the Tree:** Write a recursive function `evaluate_tree(node, x)`.
    * *Base Cases (Leaves):* If the `node.value` is the string `'x'`, return the numeric value of `x`. If `node.value` is a number (e.g., an `int` or `float`), return that number.
    * *Recursive Step (Internal Nodes):* If the `node.value` is an operator (`'+'`, `'-'`, `'*'`, `'/'`), recursively call `evaluate_tree` on `node.left` and `node.right`.
    * Apply the operator to the left and right results and return the final mathematical answer. *(Hint: Check for division by zero! If the operator is `'/'` and the right result is `0`, return `1` as a safe fallback to prevent crashes).*
* **2. Calculate Fitness (MSE):** Write `calculate_mse(root_node, dataset)`.
    * Initialize a `total_error` variable to 0.
    * Loop through every `(x_val, y_true)` tuple in the provided `dataset`.
    * Call your `evaluate_tree(root_node, x_val)` function to get `y_pred`.
    * Calculate the squared error: `(y_true - y_pred) ** 2`, and add it to `total_error`.
    * Return `total_error / len(dataset)`.

In [ ]:
def evaluate_tree(node, x):
    """Recursively evaluate a syntax tree for a given input x."""
    # Leaves: the variable 'x' or a numeric constant
    if node.value == 'x':
        return x
    if isinstance(node.value, (int, float)):
        return node.value

    # Internal node: evaluate children, then apply the operator
    left_val = evaluate_tree(node.left, x)
    right_val = evaluate_tree(node.right, x)

    if node.value == '+':
        return left_val + right_val
    elif node.value == '-':
        return left_val - right_val
    elif node.value == '*':
        return left_val * right_val
    elif node.value == '/':
        if right_val == 0:  # guard against divide-by-zero
            return 1
        return left_val / right_val


def calculate_mse(root_node, dataset):
    """Fitness function: mean squared error of the tree's predictions over the dataset."""
    total_error = 0
    for x_val, y_true in dataset:
        y_pred = evaluate_tree(root_node, x_val)
        total_error += (y_true - y_pred) ** 2
    return total_error / len(dataset)

### The Test Harness
*Run this code to test if your execution engine correctly traverses the syntax tree and calculates the MSE.*

In [ ]:
# MILESTONE 1 TEST HARNESS

# 1. Test the evaluation engine with a single input
test_x = 4
predicted_y = evaluate_tree(test_root, test_x)
print(f"Tree Evaluation for x={test_x}: Output is {predicted_y}")

assert predicted_y == 9, f"Failure: Expected 9, but got {predicted_y}"

# 2. Test the Fitness (MSE) function
tree_mse = calculate_mse(test_root, dataset)
print(f"Mean Squared Error (Fitness) for the Test Tree: {tree_mse}")

assert tree_mse == 0.0, f"Failure: The Test Tree perfectly matches the dataset, so MSE should be 0.0!"

# Let's test a "bad" tree: f(x) = x * x
bad_tree = Node('*', left=Node('x'), right=Node('x'))
bad_mse = calculate_mse(bad_tree, dataset)
print(f"Mean Squared Error (Fitness) for a Bad Tree (x * x): {bad_mse}")

assert bad_mse > 0.0, "Failure: A bad tree should have a high MSE."

print("\nSUCCESS! Symbolic Regression execution and fitness engines are operational.")

### Architect's Audit (Milestone 1)

1. **Tree vs. Array:** In Module 2, we used fixed-length arrays (e.g., `[2, 0, 3, 1]`) to represent drone configurations. Why is a hierarchical Syntax Tree mathematically necessary for Genetic Programming and Symbolic Regression compared to a fixed array? What flexibility does the tree provide?

Question One:

A fixed-length array only functions when the solution's shape is predetermined—Module 2 consistently searches for exactly N drone positions. Since Symbolic Regression isn't aware beforehand of how many terms or the depth of the correct formula, the representation must be adaptable, capable of growing, shrinking, and nesting. A syntax tree captures both structure and content in one form, allowing subtree swaps to replace entire valid sub-expressions without disrupting positional integrity, unlike cutting an array at a random index which can corrupt data. This tree flexibility allows it to depict complex, variable-sized expressions, unlike an array that is confined to a fixed length and shape.

## Milestone 2: Set Up

*Run this cell to load our tree-manipulation helper functions. Navigating trees can be tricky, so we have provided `copy_tree` (to prevent destroying the original parents), `get_all_nodes` (to help you randomly pick a crossover point), and `tree_to_string` (so you can read the equations).*


In [ ]:
# MILESTONE 2 SETUP - RUN THIS CELL FIRST
import random

def copy_tree(node):
    """Recursively creates a deep copy of a tree."""
    if node is None:
        return None
    return Node(node.value, copy_tree(node.left), copy_tree(node.right))

def get_all_nodes(node, nodes_list=None):
    """Returns a list of all Node objects in the tree."""
    if nodes_list is None:
        nodes_list = []
    if node is not None:
        nodes_list.append(node)
        get_all_nodes(node.left, nodes_list)
        get_all_nodes(node.right, nodes_list)
    return nodes_list

def tree_to_string(node):
    """Converts a Syntax Tree back into a readable math equation."""
    if node is None:
        return ""
    if node.left is None and node.right is None:
        return str(node.value)
    return f"({tree_to_string(node.left)} {node.value} {tree_to_string(node.right)})"

# Parent A: ((x * 2) + 1)
parent_A = Node('+', left=Node('*', left=Node('x'), right=Node(2)), right=Node(1))

# Parent B: ((x - 4) * 5)
parent_B = Node('*', left=Node('-', left=Node('x'), right=Node(4)), right=Node(5))

print("Parent A Equation:", tree_to_string(parent_A))
print("Parent B Equation:", tree_to_string(parent_B))
print("Helper functions loaded.")

## Milestone 2: Genetic Operators (Crossover & Mutation)
**Objective:** In standard Genetic Algorithms, reproduction involves splicing lists and flipping bits. In Genetic Programming, reproduction involves cutting and mutating **branches of a syntax tree**.

* **Subtree Crossover:** The algorithm selects a random node in Parent A, selects a random node in Parent B, and swaps the subtrees rooted at those nodes.
* **Point Mutation:** The algorithm selects a random node in a single chromosome and randomly changes its value (e.g., changing a `+` to a `*`, or changing a `2` to a `5`).

### The Architect's Blueprint
*Write the `subtree_crossover(parent1, parent2)` and `mutate_tree(root_node, mutation_rate)` functions.*

* **1. Crossover:** * Create `child1` and `child2` by passing `parent1` and `parent2` into `copy_tree()`.
    * Use `get_all_nodes()` to get a list of all nodes in `child1`. Use `random.choice()` to pick one random node. Call it `node_A`. Do the same for `child2` to get `node_B`.
    * **The Swap Trick:** Swap `node_A.value` with `node_B.value`, `node_A.left` with `node_B.left`, and `node_A.right` with `node_B.right`. Return `child1` and `child2`.
* **2. Mutation:** * Use `get_all_nodes(root_node)` to get a list of all nodes in the tree.
    * Loop through every node in that list. Roll a random float between `0.0` and `1.0`.
    * If the random number is less than your `mutation_rate`:
        * Check what kind of node it is. If it is an operator (`node.left` and `node.right` are NOT None), randomly change `node.value` to one of `['+', '-', '*', '/']`.
        * If it is a terminal leaf (`node.left` and `node.right` ARE None), randomly change `node.value` to either `'x'` or a random integer between `1` and `10`.
    * Return the mutated `root_node`.

In [ ]:
def subtree_crossover(parent1, parent2):
    """Swap a random subtree between two parent trees to produce two children."""
    child1 = copy_tree(parent1)
    child2 = copy_tree(parent2)

    node_a = random.choice(get_all_nodes(child1))
    node_b = random.choice(get_all_nodes(child2))

    # Swap contents in place so each subtree gets grafted onto the other tree
    node_a.value, node_b.value = node_b.value, node_a.value
    node_a.left, node_b.left = node_b.left, node_a.left
    node_a.right, node_b.right = node_b.right, node_a.right

    return child1, child2


def mutate_tree(root_node, mutation_rate):
    """Randomly perturb node values in place, each with probability mutation_rate."""
    for node in get_all_nodes(root_node):
        if random.random() < mutation_rate:
            if node.left is not None and node.right is not None:  # operator node
                node.value = random.choice(['+', '-', '*', '/'])
            else:  # terminal leaf
                node.value = random.choice(['x', random.randint(1, 10)])
    return root_node

### The Test Harness

In [ ]:
# MILESTONE 2 TEST HARNESS
print("--- GENETIC REPRODUCTION RESULTS ---")

# 1. Test Crossover
child_A, child_B = subtree_crossover(parent_A, parent_B)
print(f"Child A (After Crossover): {tree_to_string(child_A)}")
print(f"Child B (After Crossover): {tree_to_string(child_B)}")

assert tree_to_string(child_A) != tree_to_string(parent_A), "Failure: Child A identical to Parent A."

# 2. Test Mutation (Using 100% mutation rate to force a change)
mutated_A = mutate_tree(copy_tree(parent_A), mutation_rate=1.0)
print(f"Parent A (Before Mutation): {tree_to_string(parent_A)}")
print(f"Parent A (After 100% Mutation):  {tree_to_string(mutated_A)}")

assert tree_to_string(mutated_A) != tree_to_string(parent_A), "Failure: Mutation did not alter the tree."
print("\nSUCCESS! Genetic operators are operational.")

### Architect's Audit (Milestone 2)

1. **The Building Blocks of Math:** In Lab 2, crossover simply meant cutting an array at an index. In this lab, we swap entire, multi-layered subtrees. Conceptually, what are we attempting to preserve when we swap a full subtree instead of just randomly mixing individual nodes?
2. **The Necessity of Mutation:** Look at your `mutate_tree` function. Suppose you disabled mutation entirely and *only* ran subtree crossover for 1,000 generations. Why might the algorithm be mathematically incapable of ever finding our hidden formula `y = (x * 2) + 1`? What specific limitation does crossover have that mutation fixes?

Question One:

Swapping an entire subtree retains the meaningful partial "building block" it represents — for example, `(x * 2)` already performs a useful calculation, and integrating it unchanged into another tree preserves that meaning. In contrast, shuffling individual node values would disrupt the connection between operators and their operands, leading to the loss of any partially correct subexpression already found. This concept is similar to maintaining beneficial gene combinations in a standard genetic algorithm, but here, the "gene" is a complete functional expression instead of a single value.

Question Two:

Crossover can only recombine existing values and operators from the current population; it does not introduce new ones that weren't originally present. For example, if the constant `1` or the `*` operator is absent in the first generation, crossover alone cannot generate the expression `(x * 2) + 1`, as that component doesn't exist to be swapped in. Mutation is necessary to introduce truly new values and operators into the gene pool. Without mutation, the search remains limited to rearranging existing elements, preventing exploration of the complete space of possible formulas.

## Milestone 3: Set Up

Run this cell first.

*We have provided this generator to help you create your initial population (the "Grow" method from Lecture 3.1) and a printing function so you can visualize the trees.*

In [ ]:
# MILESTONE 3 SETUP - RUN THIS CELL FIRST
def print_tree(node, prefix="", is_last=True, is_root=True):
    """
    Visually prints the Syntax Tree in a 2D hierarchical format using ASCII branches.
    """
    if node is None:
        return

    # Print the current node with the appropriate branch symbols
    if is_root:
        print(str(node.value))
    else:
        print(prefix + ("└── " if is_last else "├── ") + str(node.value))

    # Prepare the indentation prefix for the children
    if not is_root:
        child_prefix = prefix + ("    " if is_last else "│   ")
    else:
        child_prefix = ""

    # Recursively print left and right children
    if node.left or node.right:
        print_tree(node.left, child_prefix, is_last=False, is_root=False)
        print_tree(node.right, child_prefix, is_last=True, is_root=False)

# --- Example Usage ---
# Let's test it on our (x * 2) + 1 tree from Milestone 1
node_x = Node('x')
node_2 = Node(2)
node_mult = Node('*', left=node_x, right=node_2)
node_1 = Node(1)
test_root = Node('+', left=node_mult, right=node_1)

print("Visualizing Tree Structure:")
print_tree(test_root)

def generate_random_tree(depth, max_depth):
    """Recursively generates a random syntax tree."""
    # If we hit max depth, we must return a terminal (leaf)
    if depth >= max_depth or (depth > 0 and random.random() < 0.3):
        terminal = random.choice(['x', random.randint(1, 10)])
        return Node(terminal)

    # Otherwise, return an operator with two child branches
    operator = random.choice(['+', '-', '*', '/'])
    left_child = generate_random_tree(depth + 1, max_depth)
    right_child = generate_random_tree(depth + 1, max_depth)
    return Node(operator, left_child, right_child)

# The dataset we are trying to solve!
# Hidden formula: y = (x * 2) + 1
dataset = [(1, 3), (2, 5), (3, 7), (4, 9), (5, 11)]
print("Random Tree Generator loaded.")

## Milestone 3: The GP Lifecycle (Solving Symbolic Regression)
**Objective:** It is time to let the equations evolve. You will implement the main loop to discover the hidden mathematical formula driving our dataset.

### The Architect's Blueprint
*Write the `run_genetic_programming(pop_size, generations, mutation_rate)` function.*

* **1. Initialization:** Create a `population` list of size `pop_size`. Fill it with trees generated by `generate_random_tree(depth=0, max_depth=3)`.
* **2. The Loop:** Run a loop for `generations`. Inside the loop:
    * Evaluate the fitness (MSE) of every tree in the population using your `calculate_mse(tree, dataset)` function.
    * Sort the population from lowest MSE (best) to highest MSE (worst).
    * *Early Stopping:* If the best tree has an MSE of `0.0` (or extremely close, like `< 0.01`), `break` the loop!
    * **Next Generation:** Create an empty `new_population` list. Pass the best 10% of trees directly into the new population (Elitism).
    * **Reproduce:** While `new_population` is not full:
        * Select two parents from the top 50% of the current population using `random.choice()`.
        * Perform `subtree_crossover` to get two children.
        * Perform `mutate_tree` on both children.
        * Append them to `new_population`.
    * Update `population = new_population`.
* **3. Return:** Return the single best tree found and its MSE.

In [ ]:
def run_genetic_programming(pop_size, generations, mutation_rate):
    """Evolve a population of syntax trees toward the hidden formula; returns the best tree and its MSE."""
    population = [generate_random_tree(depth=0, max_depth=3) for _ in range(pop_size)]

    best_tree, best_mse = None, float('inf')

    for generation in range(generations):
        # Rank the population by fitness (lower MSE is better)
        scored = sorted(population, key=lambda tree: calculate_mse(tree, dataset))
        best_tree = scored[0]
        best_mse = calculate_mse(best_tree, dataset)

        if best_mse < 0.01:  # early stopping: close enough to a perfect fit
            break

        # Elitism: carry the best 10% forward unchanged
        elite_count = max(1, pop_size // 10)
        new_population = scored[:elite_count]

        # Reproduce from the top 50% until the new population is full
        mating_pool = scored[:max(2, pop_size // 2)]
        while len(new_population) < pop_size:
            parent1 = random.choice(mating_pool)
            parent2 = random.choice(mating_pool)
            child1, child2 = subtree_crossover(parent1, parent2)
            new_population.append(mutate_tree(child1, mutation_rate))
            if len(new_population) < pop_size:
                new_population.append(mutate_tree(child2, mutation_rate))

        population = new_population

    return best_tree, best_mse

### The Test Harness

In [ ]:
# MILESTONE 3 TEST HARNESS
print("Initializing Synthetica GP Engine...")

best_equation, best_mse = run_genetic_programming(
    pop_size=200,
    generations=50,
    mutation_rate=0.1
)

print("\n--- EVOLUTION COMPLETE ---")
print(f"Best Equation Found: {tree_to_string(best_equation)}")
print(f"Final Mean Squared Error: {best_mse}")

if best_mse < 0.1:
    print("SUCCESS: The Symbolic Regression engine cracked the hidden formula!")
else:
    print("WARNING: The algorithm failed to converge on the perfect formula.")

In [ ]:
print_tree(best_equation)

### Architect's Audit (Milestone 3)

1. **The Threat of Bloat:** Run your Test Harness a few times. Did your final winning equation output a clean `((x * 2) + 1)`, or did it output something messy like `(((x * 2) + 1) + (0 * x))`? Based on Lecture 3.2, explain the concept of "Bloat" in Genetic Programming. Why does random subtree crossover naturally cause trees to grow unnecessarily massive over time?

Question One:

Running the test harness repeatedly rarely produces the clean `(x * 2) + 1` — more often it finds something like `(((x + x) + (6 / 6)) - (x - x))` or an even deeper nested equivalent, all of which compute the right answer but carry redundant sub-expressions that just evaluate to `0` or `1` and cancel out. This is "Bloat": the tendency for GP individuals to grow steadily larger and more complex over generations without any corresponding gain in fitness, since MSE only measures output accuracy and has no notion of expression size. Subtree crossover drives this because once a tree reaches zero (or near-zero) error, splicing in an "inert" subtree — one that evaluates to `0` via addition/subtraction or `1` via multiplication/division, like `(x - x)` or `(x / x)` — doesn't change the fitness at all, so these neutral, larger variants survive selection exactly as well as the compact one and keep accumulating generation after generation.

## The Summary Audit

To complete the Lab, you must submit your final Colab Notebook containing your code along with brief 3-4 sentence answers to the following questions synthesizing the module:

1. **Pruning the Trees:** If you were writing the `calculate_mse` fitness function for a production environment, propose one mathematical penalty you could add to the MSE calculation to naturally encourage the algorithm to evolve *shorter* equations.

Question One:

I suggest adding a parsimony penalty to the fitness score: fitness = calculate_mse(root_node, dataset) + lambda_penalty * len(get_all_nodes(root_node)), where lambda_penalty is a small coefficient (around 0.001-0.01) set to only break ties between similarly accurate trees, not to override actual accuracy improvements. Since get_all_nodes already counts all nodes, this size penalty is inexpensive to compute and considers every extra node — including inert subtrees like (x - x) or (x / x) that cause bloat — as a real cost rather than a freebie. When ranking the population using this combined score, a bloated tree with the same MSE as a smaller one will score worse and be less likely to survive or mate, directly counteracting the neutral drift effect explained in the Milestone 3 bloat discussion.